<a href="https://colab.research.google.com/github/MeetuKumari1/ml_productionization/blob/main/gender_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Project Name**    -

$\color{red}{\text{Gender Classification Model}}$


##### **Project Type**    - Classification
##### **Contribution**    - Individual
##### **Team Member 1**-  $\color{green}{\text{Meetu Kumari}}$


# **GitHub Link -**

 https://github.com/MeetuKumari1/ml_productionization.git

Project Title:

Voyage Analytics: Integrating MLOps in Travel Productionization of ML Systems

### Brief overview of users datasets

* code: User identifier.
* company: Associated company.
* name: Name of the user.
* gender: Gender of the user.
* age: Age of the user.

Project Objectives

Deploy a classification model to categorize a user's gender.

## Let's Begin !

### 1. Know Your Data

In [ ]:
#pip install mlflow

Import Libraries

In [ ]:
import json
import os
import joblib
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression, SGDClassifier, RidgeClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
# Dataset loading
from pathlib import Path
import zipfile

# zip_path = Path("C:/Users/Documents/ml_productionization/travel_capstone.zip")
# extract_dir = zip_path.parent

# with zipfile.ZipFile(zip_path, "r") as zf:
#     zf.extractall(extract_dir)

# print(f"Extracted to: {extract_dir}")

## for google drive (colab)
from google.colab import drive
drive.mount('/content/drive')
zip_path = Path("/content/drive/MyDrive/AlmaBetter/AB_Specialisation_Track/capstone_project/ML_engineering/ml_productionization/travel_capstone.zip")
extract_dir = zip_path.parent
with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(extract_dir)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# load flight dataset
users_df = pd.read_csv(
    "/content/drive/MyDrive/AlmaBetter/AB_Specialisation_Track/capstone_project/ML_engineering/ml_productionization/travel_capstone/users.csv"
)

In [ ]:
users_df.head()

,code,company,name,gender,age
0,0,4You,Roy Braun,male,21
1,1,4You,Joseph Holsten,male,37
2,2,4You,Wilma Mcinnis,female,48
3,3,4You,Paula Daniel,female,23
4,4,4You,Patricia Carson,female,44


In [ ]:
users_df.tail()

,code,company,name,gender,age
1335,1335,Umbrella LTDA,Albert Garroutte,male,23
1336,1336,Umbrella LTDA,Kim Shores,female,40
1337,1337,Umbrella LTDA,James Gimenez,male,28
1338,1338,Umbrella LTDA,Viola Agosta,female,52
1339,1339,Umbrella LTDA,Paul Rodriguez,male,35


In [ ]:
# Dataset Rows & Columns count
print(f"Total number of rows: {users_df.shape[0]}")
print(f"Total number of columns: {users_df.shape[1]}")

Total number of rows: 1340
Total number of columns: 5


#### Dataset Information

In [ ]:
#Dataset Info
users_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1340 entries, 0 to 1339
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   code     1340 non-null   int64 
 1   company  1340 non-null   object
 2   name     1340 non-null   object
 3   gender   1340 non-null   object
 4   age      1340 non-null   int64 
dtypes: int64(2), object(3)
memory usage: 52.5+ KB


In [ ]:
#dataset duplicate values
users_df.duplicated().value_counts()

False    1340
Name: count, dtype: int64

In [ ]:
#missing values or null values
users_df.isnull().sum()

code       0
company    0
name       0
gender     0
age        0
dtype: int64

#### What did you know about your dataset?

This dataset has 1340 rows and 5 columns. It contains no missing values and no duplicate rows. The feature mix includes one integer identifiers (`userCode`), 3 categorical columns (`company`, `name`, `gender`). The in-memory size is ~52.5 kB. For the classification task, `gender` is the target variable and the remaining fields can be used as features after encoding and date parsing.

### 2. Understanding the Variables

##### Check Unique Values for each variable.

In [ ]:
#dataset description
users_df.describe()

,code,age
count,1340.000000,1340.000000
mean,669.500000,42.742537
std,386.968991,12.869779
min,0.000000,21.000000
25%,334.750000,32.000000
50%,669.500000,42.000000
75%,1004.250000,54.000000
max,1339.000000,65.000000


In [ ]:
# description for only categorical columns
users_df.describe(include=['object'])

,company,name,gender
count,1340,1340,1340
unique,5,1338,3
top,4You,Charlotte Johnson,male
freq,453,2,452


In [ ]:
# Check Unique Values for each variable.
users_df.nunique()

code       1340
company       5
name       1338
gender        3
age          45
dtype: int64

In [ ]:
## Unique Values
for col in users_df.columns:
    print(f"Unique values for column_name '{col}' is: \n{users_df[col].unique()}\n")

Unique values for column_name 'code' is: 
[   0    1    2 ... 1337 1338 1339]

Unique values for column_name 'company' is: 
['4You' 'Monsters CYA' 'Wonka Company' 'Acme Factory' 'Umbrella LTDA']

Unique values for column_name 'name' is: 
['Roy Braun' 'Joseph Holsten' 'Wilma Mcinnis' ... 'James Gimenez'
 'Viola Agosta' 'Paul Rodriguez']

Unique values for column_name 'gender' is: 
['male' 'female' 'none']

Unique values for column_name 'age' is: 
[21 37 48 23 44 47 46 41 35 36 61 53 56 25 65 22 51 60 64 49 62 59 40 34
 27 42 24 54 28 55 39 38 32 29 52 57 31 45 30 43 58 63 50 26 33]



I will evaluate multiple classification models, compare their accuracy, and select the best-performing model for deployment.

In [ ]:
# Install xgboost if not already installed
#%pip install -q xgboost

In [ ]:
NUMERIC_FEATURES = ["age", "name_length", "vowel_ratio"]
CATEGORICAL_FEATURES = [
    "company",
    "first_char",
    "last_char",
    "last_2",
    "last_3",
    "first_token",
    "last_token",
]

preprocess = ColumnTransformer(
    transformers=[
        ("num", MinMaxScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
        (
            "name",
            TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2),
            "name",
        ),
    ]
)

In [ ]:
FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES + ["name"]
TARGET_COLUMN = "gender"

In [ ]:
def build_name_features(frame: pd.DataFrame) -> pd.DataFrame:
    if "name" not in frame.columns:
        raise ValueError("Missing 'name' column in dataset.")

    data = frame.copy()
    name_series = data["name"].fillna("").astype(str).str.strip()
    name_lower = name_series.str.lower()

    data["name_length"] = name_lower.str.len()
    vowel_count = name_lower.str.count("[aeiou]")
    data["vowel_ratio"] = np.where(data["name_length"] > 0, vowel_count / data["name_length"], 0.0)

    data["first_char"] = name_lower.str[:1]
    data["last_char"] = name_lower.str[-1:]
    data["last_2"] = name_lower.str[-2:]
    data["last_3"] = name_lower.str[-3:]

    tokens = name_lower.str.split()
    data["first_token"] = tokens.str[0].fillna("")
    data["last_token"] = tokens.str[-1].fillna("")

    return data

users_df = build_name_features(users_df)

In [ ]:
if users_df[FEATURE_COLUMNS + [TARGET_COLUMN]].isna().any().any():
    raise ValueError("Dataset contains missing values in required columns.")
X = users_df[FEATURE_COLUMNS]
y = users_df[TARGET_COLUMN]


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42,
                                    stratify=y)

In [ ]:
models = {
    "LogisticRegression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42,
    ),
    "LinearSVC": LinearSVC(class_weight="balanced", random_state=42),
    "SGDClassifier_LogLoss": SGDClassifier(
        loss="log_loss",
        class_weight="balanced",
        max_iter=2000,
        tol=1e-3,
        random_state=42,
    ),
    "RidgeClassifier": RidgeClassifier(class_weight="balanced", random_state=42),
    "MultinomialNB": MultinomialNB(),
    "ComplementNB": ComplementNB(),
}

In [ ]:
results = []
best_model_name = None
best_accuracy = -1.0
best_pipeline = None

for model_name, model in models.items():
    pipeline = Pipeline(steps=[("preprocess", preprocess), ("model", model)])
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_val)
    acc = float(accuracy_score(y_val, preds))
    f1_macro = float(f1_score(y_val, preds, average="macro"))

    results.append(
        {
            "model_name": model_name,
            "accuracy": acc,
            "f1_macro": f1_macro,
        }
    )

    with mlflow.start_run(run_name=f"{model_name}_gender"):
        mlflow.log_param("model_name", model_name)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_macro", f1_macro)

    if acc > best_accuracy:
        best_accuracy = acc
        best_model_name = model_name
        best_pipeline = pipeline

results_df = pd.DataFrame(results).sort_values(by="accuracy", ascending=False)
results_df

,model_name,accuracy,f1_macro
1,LinearSVC,0.541045,0.537342
2,SGDClassifier_LogLoss,0.541045,0.545062
3,RidgeClassifier,0.529851,0.525157
5,ComplementNB,0.503731,0.490735
0,LogisticRegression,0.500000,0.497421
4,MultinomialNB,0.500000,0.491346


In [ ]:
DATA_PATH = Path("travel_capstone/users.csv")
MODEL_PATH = Path("artifacts/user_gender_model.joblib")
METADATA_PATH = Path("artifacts/user_gender_metadata.json")


In [ ]:
with mlflow.start_run(run_name="best_gender_model"):
    mlflow.log_param("best_model_name", best_model_name)
    mlflow.log_metric("best_accuracy", best_accuracy)

    MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(best_pipeline, MODEL_PATH)

    metadata = {
        "model_name": best_model_name,
        "accuracy": best_accuracy,
        "features": FEATURE_COLUMNS,
    }
    METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

    mlflow.sklearn.log_model(best_pipeline, artifact_path="model")
    mlflow.log_artifact(str(METADATA_PATH))

print(results_df)
print(f"Saved best model to {MODEL_PATH}")
print(f"Best validation accuracy: {best_accuracy:.4f}")


2026/02/02 18:57:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


              model_name  accuracy  f1_macro
1              LinearSVC  0.541045  0.537342
2  SGDClassifier_LogLoss  0.541045  0.545062
3        RidgeClassifier  0.529851  0.525157
5           ComplementNB  0.503731  0.490735
0     LogisticRegression  0.500000  0.497421
4          MultinomialNB  0.500000  0.491346
Saved best model to artifacts\user_gender_model.joblib
Best validation accuracy: 0.5410
